### External

In [24]:
import kagglehub
import os
import zipfile
import os
from pathlib import Path
import shutil


In [25]:


# Descargar dataset
path = kagglehub.dataset_download("kaustubhdhote/human-faces-dataset")

# Ruta destino
destino = Path("../data/external")

destino.mkdir(parents=True, exist_ok=True)

# Copiar archivos al proyecto
shutil.copytree(path, destino, dirs_exist_ok=True)

print("Archivos copiados correctamente")

Archivos copiados correctamente


### Raw

In [26]:
from pathlib import Path
from PIL import Image
import shutil

In [27]:
# ========= 1. Definir rutas =========

PROJECT_ROOT = Path().resolve().parent
EXTERNAL_PATH = PROJECT_ROOT / "data" / "external" / "human faces dataset"
RAW_PATH = PROJECT_ROOT / "data" / "raw"

AI_SRC = EXTERNAL_PATH / "ai-generated images"
REAL_SRC = EXTERNAL_PATH / "real images"

AI_DST = RAW_PATH / "ai"
REAL_DST = RAW_PATH / "real"

AI_DST.mkdir(parents=True, exist_ok=True)
REAL_DST.mkdir(parents=True, exist_ok=True)

# ========= 2. Función para validar imágenes =========

def process_images(src_folder, dst_folder):
    total = 0
    valid = 0
    corrupted = 0
    
    for img_path in src_folder.glob("*"):
        total += 1
        
        try:
            with Image.open(img_path) as img:
                img.verify()  # verifica que no esté corrupta
            
            shutil.copy(img_path, dst_folder / img_path.name)
            valid += 1
            
        except Exception:
            corrupted += 1
    
    return total, valid, corrupted

# ========= 3. Procesar datasets =========

ai_total, ai_valid, ai_corrupted = process_images(AI_SRC, AI_DST)
real_total, real_valid, real_corrupted = process_images(REAL_SRC, REAL_DST)

# ========= 4. Reporte simple =========

print("===== DATASET REPORT =====")
print(f"AI Images     → Total: {ai_total}, Válidas: {ai_valid}, Corruptas: {ai_corrupted}")
print(f"Real Images   → Total: {real_total}, Válidas: {real_valid}, Corruptas: {real_corrupted}")
print(f"Total limpio en RAW: {ai_valid + real_valid}")

===== DATASET REPORT =====
AI Images     → Total: 4630, Válidas: 4630, Corruptas: 0
Real Images   → Total: 5000, Válidas: 5000, Corruptas: 0
Total limpio en RAW: 9630


In [28]:
print("\nBalance de clases:")
print(f"% AI   : {ai_valid / (ai_valid + real_valid) * 100:.2f}%")
print(f"% Real : {real_valid / (ai_valid + real_valid) * 100:.2f}%")


Balance de clases:
% AI   : 48.08%
% Real : 51.92%


### Interim

In [34]:
import os
import random
import shutil
from pathlib import Path


In [36]:

# Para reproducibilidad
random.seed(42)

PROJECT_ROOT = Path().resolve().parent

RAW_PATH = PROJECT_ROOT / "data" / "raw"
INTERIM_PATH = PROJECT_ROOT / "data" / "interim"

train_split = 0.7
val_split = 0.15
test_split = 0.15

for split in ["train", "val", "test"]:
    for cls in ["ai", "real"]:
        (INTERIM_PATH / split / cls).mkdir(parents=True, exist_ok=True)

def split_class_images(class_name):
    source_dir = RAW_PATH / class_name
    images = os.listdir(source_dir)
    
    random.shuffle(images)
    
    total = len(images)
    train_size = int(total * train_split)
    val_size = int(total * val_split)
    
    train_imgs = images[:train_size]
    val_imgs = images[train_size:train_size + val_size]
    test_imgs = images[train_size + val_size:]
    
    splits = {
        "train": train_imgs,
        "val": val_imgs,
        "test": test_imgs
    }
    
    for split_name, img_list in splits.items():
        for img in img_list:
            shutil.copy(
                source_dir / img,
                INTERIM_PATH / split_name / class_name / img
            )
    
    return total, len(train_imgs), len(val_imgs), len(test_imgs)

ai_stats = split_class_images("ai")
real_stats = split_class_images("real")

print("===== SPLIT REPORT =====")
print(f"AI    → Total: {ai_stats[0]} | Train: {ai_stats[1]} | Val: {ai_stats[2]} | Test: {ai_stats[3]}")
print(f"REAL  → Total: {real_stats[0]} | Train: {real_stats[1]} | Val: {real_stats[2]} | Test: {real_stats[3]}")

===== SPLIT REPORT =====
AI    → Total: 4630 | Train: 3241 | Val: 694 | Test: 695
REAL  → Total: 5000 | Train: 3500 | Val: 750 | Test: 750
